# Native Memory Pointer Pattern — Strands `ContextOffloader`

The companion notebook (`test_context_overflow.ipynb`) builds the Memory Pointer Pattern **by hand**: tools store large data in `agent.state` and return a pointer string. That teaches the concept from scratch.

This notebook shows the **same pattern as a first-class Strands feature**. Strands now ships `ContextOffloader` — a plugin that intercepts large tool results at execution time, stores each block in a storage backend, and leaves a small preview plus a reference in context. The offloading concern moves **out of your tools** and into the framework.

> This demo uses Strands Agents. Offloading large tool outputs and summarizing history are general agent concepts and carry over to other agent frameworks.

## Manual vs Native

| | Manual (`tools.py`) | Native (`native_tools.py`) |
|---|---|---|
| Fetch tool | Stores in `agent.state`, returns a pointer string | Ordinary function — just returns the JSON |
| Analysis tool | Receives `logs_pointer`, calls `agent.state.get()` | Ordinary function — no pointer logic |
| Who offloads | You, inside every tool | The `ContextOffloader` plugin, outside the tools |
| Retrieval | Read `agent.state` by key | `retrieve_offloaded_content(reference)` — by exact reference |

## What we test (same query, three strategies)

| Test | Strategy | What it shows |
|------|----------|---------------|
| 1 | No context management | Raw JSON enters the context window → high tokens |
| 2 | `ContextOffloader` (FileStorage) | Large results offloaded to disk → low tokens |
| 3 | `context_manager="auto"` | One line composes Summarizing + ContextOffloader |

## Setup: install dependencies

This demo needs **strands-agents 1.56.0+** (earlier versions don't have `ContextOffloader` or `context_manager="auto"`). Run the cell below once to install everything from `requirements.txt`, then restart the kernel if prompted.

In [1]:
%pip install -r requirements.txt

# Verify the installed version is new enough for the native context APIs
import importlib.metadata as _m
_v = _m.version("strands-agents")
assert tuple(int(x) for x in _v.split(".")[:2]) >= (1, 56), (
    f"strands-agents {_v} is too old. This demo needs >= 1.56.0. "
    "Re-run the install cell and restart the kernel."
)
print(f"strands-agents {_v} — OK")


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
strands-agents 1.56.0 — OK


## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any provider supported by Strands — see [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/) for configuration.

In [2]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    '⚠️ OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

Note the imports: `native_tools` provides **ordinary** log tools (no pointer logic), and `ContextOffloader` comes from `strands.vended_plugins`, and the storage backends (`LocalFileStorage`, `S3Storage`, `InMemoryStorage`) from `strands.storage` (Strands 1.56+).

In [3]:
import json, time, os, shutil

os.environ['OTEL_SDK_DISABLED'] = 'true'
import logging, warnings  # silence OpenTelemetry 'Failed to detach context' noise
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='Failed to detach context')

from dotenv import load_dotenv
from strands import Agent
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.storage import LocalFileStorage, S3Storage
from strands.vended_plugins.context_offloader import ContextOffloader

from native_tools import fetch_application_logs, count_errors_by_service

load_dotenv()

MODEL = OpenAIModel(model_id='gpt-4o-mini')
ARTIFACT_DIR = './artifacts'

# Same query for all three tests — the only variable is the context-management strategy
QUERY = (
    "Fetch 2 hours of logs for 'api-gateway', then tell me how many errors occurred "
    'and which service had the most.'
)


def count_context_tokens(agent) -> int:
    """Approximate tokens across all messages in the conversation history (chars/4)."""
    total = 0
    for msg in agent.messages:
        content = msg.get('content', [])
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict):
                    if 'text' in block:
                        total += len(block['text']) // 4
                    elif 'toolResult' in block:
                        for item in block['toolResult'].get('content', []):
                            if 'text' in item:
                                total += len(item['text']) // 4
                    elif 'toolUse' in block:
                        total += len(json.dumps(block['toolUse'].get('input', {}))) // 4
    return total


# Start from a clean artifacts directory so Test 2's file count reflects this run only
if os.path.isdir(ARTIFACT_DIR):
    shutil.rmtree(ARTIFACT_DIR)

print('✅ Setup complete!')

/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


✅ Setup complete!


---
## Test 1 — No Context Management (baseline)

Plain agent, plain tools. `fetch_application_logs` returns the full JSON dataset as a tool result, and it lands in the context window. Every subsequent model call re-sends it as input tokens.

In [4]:
agent_baseline = Agent(model=MODEL, tools=[fetch_application_logs, count_errors_by_service])

start = time.time()
agent_baseline(QUERY)
time_baseline = time.time() - start
tokens_baseline = count_context_tokens(agent_baseline)

print(f'\n⏱️  {time_baseline:.1f}s')
print(f'📊 Tokens in context: {tokens_baseline:,}')


Tool #1: fetch_application_logs

Tool #2: count_errors_by_service


### Logs Summary for 'api-gateway' (Last 2 Hours)

**Total Events:** 200  
**Total Errors:** 45  
**Error

 Rate:** 22.5%

### Errors Count by Service:
- **api-gateway:** 14 Errors
- **db-connector:** 12 Errors
- **cache-layer:** 12 Errors


- **auth-service:** 7 Errors

### Service with the Most Errors:
- **Most Errors:** api-gateway (14 Errors)
⏱️  5.1s
📊 Tokens in context: 17,517


---
## Test 2 — `ContextOffloader` Plugin (native Memory Pointer Pattern)

Same tools, unchanged. We attach a `ContextOffloader` plugin backed by `LocalFileStorage`. When a tool result exceeds `max_result_tokens`, the plugin stores it on disk and replaces it with a preview plus a reference — the raw data never stays in context.

Because `count_errors_by_service` is a **selective** tool (it computes the answer server-side and returns a small summary), the agent answers from the summary and the full logs stay offloaded. Offloader as the safety net, selective tools as the win.

In [5]:
storage = LocalFileStorage(base_dir=ARTIFACT_DIR)
agent_offload = Agent(
    model=MODEL,
    tools=[fetch_application_logs, count_errors_by_service],
    # Offload any tool result over ~800 tokens; keep a ~200-token preview in context
    plugins=[ContextOffloader(storage=storage, max_result_tokens=800, preview_tokens=200)],
)

start = time.time()
agent_offload(QUERY)
time_offload = time.time() - start
tokens_offload = count_context_tokens(agent_offload)

artifacts = [f for f in os.listdir(ARTIFACT_DIR) if not f.startswith('.')] if os.path.isdir(ARTIFACT_DIR) else []

print(f'\n⏱️  {time_offload:.1f}s')
print(f'📊 Tokens in context: {tokens_offload:,}')
print(f'📦 Artifacts offloaded to {ARTIFACT_DIR}/: {len(artifacts)} file(s) — retrievable by reference')


Tool #1: fetch_application_logs



Tool #2: count_errors_by_service


Over the last 2 hours for the 'api-gateway' application:

- A total of **60 errors** occurred.
- The service with the most errors

 is the **cache-layer**, which had **17 errors**.
⏱️  3.1s
📊 Tokens in context: 470
📦 Artifacts offloaded to ./artifacts/: 1 file(s) — retrievable by reference


### Test 2b — Recover the data by its exact reference (the memory pointer)

The offloader put a short **reference** into the context in place of the data. That reference *is* the memory pointer. Here we pull it out of the conversation and use it to read the full dataset back from storage — byte for byte, by exact id. (The agent does this automatically via `retrieve_offloaded_content(reference)` when it needs the data; we do it by hand to make the pointer visible.)

In [6]:
# The offloader replaced the big result with a short REFERENCE (the memory pointer).
# Pull that reference out of the conversation, then read the full data back from
# storage by that exact reference. (The agent does this automatically via the
# retrieve_offloaded_content tool; we do it by hand to make the pointer visible.)
import asyncio, concurrent.futures, re

def run_async(coro):
    """Await a coroutine to completion, even inside Jupyter's running loop."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()

def find_reference(agent, storage):
    """Find the storage reference the ContextOffloader left in the conversation.

    The offloader writes a preview like '[Stored references:] offloader/call_..._0
    (json, N bytes)'. We match that key; if the preview shape changes, we fall
    back to whatever key exists in the storage backend.
    """
    pat = re.compile(r'offloader/[\w.\-]+')
    for msg in agent.messages:
        for block in msg.get('content', []):
            if isinstance(block, dict) and 'toolResult' in block:
                for item in block['toolResult'].get('content', []):
                    m = pat.search(item.get('text', ''))
                    if m:
                        return m.group(0)
    keys = run_async(storage.list())          # fallback: ask the backend
    return keys[0] if keys else None

reference = find_reference(agent_offload, storage)
print(f'Pointer left in context: {reference}')

# Read the full dataset back from the SAME storage by that reference (async in 1.56).
if reference:
    data = run_async(storage.read(reference))
    print(f'storage.read(reference) -> {len(data):,} bytes recovered verbatim by exact reference')
    print('The full dataset never re-entered the context window.')
else:
    print('No offload reference found — the result may have been under the threshold this run.')


Pointer left in context: offloader/call_ryNt5noyq0CpCPTUJvxngnjP_0
storage.read(reference) -> 74,684 bytes recovered verbatim by exact reference
The full dataset never re-entered the context window.


---
## Test 3 — `context_manager="auto"` (one-line setup)

For most multi-turn agents you do not need to wire up offloading and summarization separately. Passing `context_manager="auto"` composes, with benchmark-validated defaults:

- `SummarizingConversationManager` (summarizes old history instead of dropping it, with proactive compression)
- `ContextOffloader` (in-memory storage) for large tool results

Your own `conversation_manager` or `plugins`, if provided, take precedence.

In [7]:
agent_auto = Agent(
    model=MODEL,
    tools=[fetch_application_logs, count_errors_by_service],
    context_manager='auto',
)

start = time.time()
agent_auto(QUERY)
time_auto = time.time() - start
tokens_auto = count_context_tokens(agent_auto)

print(f'\n⏱️  {time_auto:.1f}s')
print(f'📊 Tokens in context: {tokens_auto:,}')
print(f'⚙️  Composed: {type(agent_auto.conversation_manager).__name__} + ContextOffloader (in-memory)')


Tool #1: fetch_application_logs



Tool #2: count_errors_by_service


In the last 2 hours, there were a total of **49 errors** logged for the 'api-gateway' application. The service

 with the most errors was the **auth-service**, which had **16 errors**. 

Here's a breakdown of errors by service:
- **auth-service**: 16 errors
- **db-connector**

: 12 errors
- **cache-layer**: 11 errors
- **api-gateway**: 10 errors
⏱️  2.8s
📊 Tokens in context: 966
⚙️  Composed: NullConversationManager + ContextOffloader (in-memory)


---
## Comparison

In [8]:
rows = [
    ('1 — No management', tokens_baseline, time_baseline),
    ('2 — ContextOffloader', tokens_offload, time_offload),
    ('3 — context_manager=auto', tokens_auto, time_auto),
]

print(f"{'Strategy':<32} {'Tokens':>10} {'Time':>8}")
print('-' * 52)
for label, tokens, elapsed in rows:
    print(f'{label:<32} {tokens:>10,} {elapsed:>6.1f}s')

best_label, best_tokens, _ = min(rows[1:], key=lambda r: r[1])
if tokens_baseline > best_tokens > 0:
    reduction = (1 - best_tokens / tokens_baseline) * 100
    print(f'\n→ Best native strategy: {best_label} — {reduction:.0f}% fewer tokens than baseline')

Strategy                             Tokens     Time
----------------------------------------------------
1 — No management                    17,517    5.1s
2 — ContextOffloader                    470    3.1s
3 — context_manager=auto                966    2.8s

→ Best native strategy: 2 — ContextOffloader — 97% fewer tokens than baseline


---
## What happens when the session closes? Persisting the pointer (local folder or S3)

Everything above kept the data **out of the LLM context** — but *where* does it actually live, and does it survive the process?

| Storage backend | Survives process exit? | Shared across machines/sessions? | Use it for |
|---|---|---|---|
| `agent.state` / `InMemoryStorage` | ❌ No — lives in RAM | ❌ No | Tests, short-lived / serverless runs |
| `LocalFileStorage` (local folder) | ✅ Yes — files on disk | ⚠️ Only on *that* machine | Local dev, debugging, single-host |
| `S3Storage` (Amazon S3) | ✅ Yes — object in a bucket | ✅ Yes — any process/host with access | Production, follow-up in another session, multi-machine |

So the honest answer to *"the session closes and I want to fetch this data from somewhere else"* is: **the pointer/reference is only as durable as the storage backend behind it.** `agent.state` and in-memory storage die with the process. `FileStorage` persists on local disk. To recall the exact data from **another session, another machine, or a serverless invocation days later**, put it in shared durable storage like **S3**.

> ⚠️ **Two different problems, don't confuse them** (per the [Strands docs](https://strandsagents.com/docs/user-guide/concepts/memory/overview/)):
> - **Memory Pointer / offloading** (this notebook) = recall the *exact same bytes* by an exact **reference**. Reference-based retrieval.
> - **`MemoryManager`** = long-term **semantic recall** of facts/preferences across sessions (`search_memory` / `add_memory`, backed by stores like Bedrock Knowledge Bases). It answers *"what do I remember about this user?"*, not *"give me back log blob #123 verbatim"*.
> - **Session management** persists the whole conversation; **context/conversation management** keeps a live session inside the window.
> For "fetch the exact log dataset from another session", you want **durable offload storage (S3)** — not the semantic MemoryManager.

In [9]:
# Persist to a LOCAL FOLDER, then recover from a *fresh* LocalFileStorage instance —
# simulating a brand-new session/process that only has the reference (the key).
import json as _json
from strands.storage import LocalFileStorage   # Strands 1.56+

PERSIST_DIR = './persisted-artifacts'
payload = _json.dumps([{'id': i, 'level': 'ERROR'} for i in range(500)]).encode()

# --- Session A: write the dataset to disk under a key (the memory pointer) ---
store_a = LocalFileStorage(base_dir=PERSIST_DIR)
run_async(store_a.write('logs-api-gateway', payload))
print("Session A wrote 'logs-api-gateway' to disk (the durable pointer).")

# --- Session B: a brand-new process would only have that key string ---
store_b = LocalFileStorage(base_dir=PERSIST_DIR)   # fresh instance, same folder
recovered = run_async(store_b.read('logs-api-gateway'))
events = _json.loads(recovered)
print(f'Session B recovered {len(events):,} events by reference '
      '— the data never re-entered the context window.')


Session A wrote 'logs-api-gateway' to disk (the durable pointer).
Session B recovered 500 events by reference — the data never re-entered the context window.


In [10]:
# Same interface, durable & SHARED across machines: swap LocalFileStorage -> S3Storage.
# S3Storage(bucket, prefix=...) has the identical async write()/read() contract.
# Guarded so this cell is safe to run without AWS credentials or a bucket.
from strands.storage import S3Storage   # Strands 1.56+

BUCKET = os.getenv('OFFLOAD_BUCKET')  # set to your bucket name to actually run it

if BUCKET:
    s3 = S3Storage(BUCKET, prefix='tool-results/')
    run_async(s3.write('logs-api-gateway', payload))
    back = run_async(s3.read('logs-api-gateway'))
    print(f'Recovered {len(_json.loads(back)):,} events from s3://{BUCKET}/ by reference.')
    # To wire it into an agent, the offloader takes the S3 backend directly:
    #   agent = Agent(model=MODEL, tools=[...],
    #                 plugins=[ContextOffloader(storage=S3Storage(BUCKET, prefix='tool-results/'))])
else:
    print('OFFLOAD_BUCKET not set — skipping the live S3 call.')
    print('In production you would do:')
    print("    from strands.storage import S3Storage")
    print("    agent = Agent(model=MODEL, tools=[...],")
    print("                  plugins=[ContextOffloader(storage=S3Storage('my-bucket', prefix='tool-results/'))])")
    print('Any session/host that can read the bucket recalls by the exact reference —')
    print("even directly with:  aws s3 cp s3://my-bucket/tool-results/<reference> -")


OFFLOAD_BUCKET not set — skipping the live S3 call.
In production you would do:
    from strands.storage import S3Storage
    agent = Agent(model=MODEL, tools=[...],
                  plugins=[ContextOffloader(storage=S3Storage('my-bucket', prefix='tool-results/'))])
Any session/host that can read the bucket recalls by the exact reference —
even directly with:  aws s3 cp s3://my-bucket/tool-results/<reference> -


---
## Summary

### Manual vs Native — when to use which

- **Manual (`agent.state`)** — when you want full control over what is stored and how it is keyed, or you are teaching the pattern from first principles.
- **`ContextOffloader`** — when you want offloading applied automatically to *any* large tool result without changing tool code, using `InMemoryStorage`, `LocalFileStorage`, or `S3Storage`.
- **`context_manager="auto"`** — the one-line default for most multi-turn agents; combines summarization and offloading.

### The core idea

Large tool outputs (logs, datasets, documents) should stay **out of the LLM context window** and be recalled **verbatim, by exact id** when needed — that is reference-based retrieval, not the semantic recall of conversational memory. Whether the data lives in `agent.state`, in memory, or on local disk, the model sees only a short pointer, never the full payload.

### References

- [Strands Context Management](https://strandsagents.com/docs/user-guide/concepts/context-management/)
- [Strands Conversation Management](https://strandsagents.com/docs/user-guide/concepts/agents/conversation-management/)
- [IBM Research: Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1)
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)